# Preprocess SSP Population and GDP Inputs

Resamples SSP population and GDP files to the 0.1 degree reference grid and normalizes
using the exact same scalars as the training data (saved in NORM_STATS_TRAIN.json by fix_data_pop_gdp.ipynb).

In [1]:
import os
import json
import numpy as np
import rasterio
import matplotlib.pyplot as plt
from rasterio.windows import from_bounds
from rasterio.enums import Resampling
from rasterio.warp import reproject, calculate_default_transform


## 1. Reference grid

In [2]:
OUTPUT_DIR = r'READY_data\inputs_normalized\ssp'
REFERENCE  = r'READY_data\labels\2024_CISI_010deg_nearest.tif'
os.makedirs(OUTPUT_DIR, exist_ok=True)

with rasterio.open(REFERENCE) as ref:
    ref_crs       = ref.crs
    ref_transform = ref.transform
    ref_width     = ref.width
    ref_height    = ref.height
    ref_profile   = ref.profile.copy()

print('Reference grid: ' + str(ref_height) + 'x' + str(ref_width) + ', res=' + str(round(abs(ref_transform[0]), 4)) + ' deg')


Reference grid: 485x570, res=0.1 deg


## 2. Functions

In [3]:
RESAMPLE_METHOD = {'gdp': Resampling.average, 'pop': Resampling.sum}

TRAIN_RAW = {
    'gdp': r'READY_data/inputs/2019_gdp_aligned_010.tif',
    'pop': r'READY_data/inputs/2020_pop_aligned_010.tif',
}

NORM_STATS_TRAIN_PATH = r'READY_data/inputs_normalized/NORM_STATS_TRAIN.json'


def resample_only(input_path, kind):
    data = np.full((ref_height, ref_width), np.nan, dtype=np.float32)
    with rasterio.open(input_path) as src:
        reproject(
            source=rasterio.band(src, 1),
            destination=data,
            dst_transform=ref_transform,
            dst_crs=ref_crs,
            resampling=RESAMPLE_METHOD[kind],
        )
    for flag in [-3.4028235e+38, -3.402823e+38, -99999, -9999, -32768]:
        data[np.isclose(data, flag, rtol=1e-5)] = np.nan
    data[data < -1e10] = np.nan
    data[np.isinf(data)] = np.nan
    data[data < 0] = np.nan
    return data


def load_training_stats(kind):
    """Load log1p mean/std from NORM_STATS_TRAIN.json (written by fix_data_pop_gdp.ipynb)."""
    with open(NORM_STATS_TRAIN_PATH) as f:
        stats = json.load(f)
    mean_val = stats[kind]['train_mean']
    std_val  = stats[kind]['train_std']
    data     = resample_only(TRAIN_RAW[kind], kind)
    max_val  = float(data[~np.isnan(data)].max())
    print('Training stats for ' + kind.upper() + ': mean=' + str(round(mean_val, 4))
          + '  std=' + str(round(std_val, 4)) + '  raw_max=' + str(round(max_val, 2)))
    return mean_val, std_val, max_val


def compute_unit_scale(file_dict, kind, train_log1p_mean):
    """Compute multiplicative scale to convert SSP values into training units."""
    n = sum(len(v) for v in file_dict.values())
    print('Computing unit scale for ' + kind.upper() + ' across ' + str(n) + ' files...')
    all_log_vals = []
    for ssp, years in file_dict.items():
        for year, path in years.items():
            d = resample_only(path, kind)
            valid = d[~np.isnan(d)]
            if len(valid):
                all_log_vals.append(np.log1p(valid))
    combined = np.concatenate(all_log_vals)
    ssp_mean = float(combined.mean())
    scale    = float(np.exp(train_log1p_mean - ssp_mean))
    print('  SSP log1p mean=' + str(round(ssp_mean, 4))
          + '  train log1p mean=' + str(round(train_log1p_mean, 4))
          + '  scale=' + str(round(scale, 6)))
    return scale


def normalize_and_save(input_path, output_path, kind, train_mean, train_std,
                        unit_scale=1.0, train_max=None):
    data = resample_only(input_path, kind)
    if unit_scale != 1.0:
        data = data * unit_scale
    if train_max is not None:
        data = np.where(~np.isnan(data), np.minimum(data, train_max), data)
    valid = ~np.isnan(data)
    if valid.sum() == 0:
        print('  WARNING: no valid data in ' + os.path.basename(input_path))
        return
    normalized = np.full_like(data, np.nan)
    normalized[valid] = (np.log1p(data[valid]) - train_mean) / (train_std + 1e-8)
    print('  ' + os.path.basename(input_path)
          + '  mean=' + str(round(float(np.nanmean(normalized)), 4))
          + '  range=[' + str(round(float(np.nanmin(normalized)), 3))
          + ', ' + str(round(float(np.nanmax(normalized)), 3)) + ']')
    profile = ref_profile.copy()
    profile.update(dtype='float32', count=1, nodata=np.nan)
    with rasterio.open(output_path, 'w', **profile) as dst:
        dst.write(normalized[np.newaxis, :, :])


print('Functions loaded.')


Functions loaded.


## 3. Process population files (15 files)

In [4]:
POP_FILES = {
    'SSP1': {2030: r'READY_data\SSP1\SSP1_2030_EU_UK_POP_01.tif',
             2050: r'READY_data\SSP1\SSP1_2050_EU_UK_POP_01.tif',
             2100: r'READY_data\SSP1\SSP1_2100_EU_UK_POP_01.tif'},
    'SSP2': {2030: r'READY_data\SSP2\SSP2_2030_EU_UK_POP_01.tif',
             2050: r'READY_data\SSP2\SSP2_2050_EU_UK_POP_01.tif',
             2100: r'READY_data\SSP2\SSP2_2100_EU_UK_POP_01.tif'},
    'SSP3': {2030: r'READY_data\SSP3\SSP3_2030_EU_UK_POP_01.tif',
             2050: r'READY_data\SSP3\SSP3_2050_EU_UK_POP_01.tif',
             2100: r'READY_data\SSP3\SSP3_2100_EU_UK_POP_01.tif'},
    'SSP4': {2030: r'READY_data\SSP4\SSP4_2030_EU_UK_POP_01.tif',
             2050: r'READY_data\SSP4\SSP4_2050_EU_UK_POP_01.tif',
             2100: r'READY_data\SSP4\SSP4_2100_EU_UK_POP_01.tif'},
    'SSP5': {2030: r'READY_data\SSP5\SSP5_2030_EU_UK_POP_01.tif',
             2050: r'READY_data\SSP5\SSP5_2050_EU_UK_POP_01.tif',
             2100: r'READY_data\SSP5\SSP5_2100_EU_UK_POP_01.tif'},
}

pop_train_mean, pop_train_std, pop_train_max = load_training_stats('pop')

for ssp, years in POP_FILES.items():
    for year, in_path in years.items():
        out_path = os.path.join(OUTPUT_DIR, 'POP_' + ssp + '_' + str(year) + '_normalized.tif')
        normalize_and_save(in_path, out_path, 'pop', pop_train_mean, pop_train_std, train_max=pop_train_max)

print('All POP files processed.')


Training stats for POP: mean=6.1368  std=3.0314  raw_max=3436316.25
  SSP1_2030_EU_UK_POP_01.tif  mean=0.1543  range=[-2.024, 2.775]
  SSP1_2050_EU_UK_POP_01.tif  mean=0.1178  range=[-2.024, 2.793]
  SSP1_2100_EU_UK_POP_01.tif  mean=-0.0036  range=[-2.024, 2.797]
  SSP2_2030_EU_UK_POP_01.tif  mean=0.1524  range=[-2.024, 2.774]
  SSP2_2050_EU_UK_POP_01.tif  mean=0.1142  range=[-2.024, 2.793]
  SSP2_2100_EU_UK_POP_01.tif  mean=0.0415  range=[-2.024, 2.821]
  SSP3_2030_EU_UK_POP_01.tif  mean=0.153  range=[-2.024, 2.774]
  SSP3_2050_EU_UK_POP_01.tif  mean=0.1215  range=[-2.024, 2.796]
  SSP3_2100_EU_UK_POP_01.tif  mean=0.1059  range=[-2.024, 2.888]
  SSP4_2030_EU_UK_POP_01.tif  mean=0.1493  range=[-2.024, 2.771]
  SSP4_2050_EU_UK_POP_01.tif  mean=0.0936  range=[-2.024, 2.774]
  SSP4_2100_EU_UK_POP_01.tif  mean=-0.0744  range=[-2.024, 2.764]
  SSP5_2030_EU_UK_POP_01.tif  mean=0.1525  range=[-2.024, 2.778]
  SSP5_2050_EU_UK_POP_01.tif  mean=0.121  range=[-2.024, 2.806]
  SSP5_2100_EU_UK_POP_

## 4. Resolution check

In [5]:
_NORM_DIR = r'READY_data\inputs_normalized\ssp'
_SSPS  = ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']
_YEARS = [2030, 2050, 2100]

# --- Raw input resolution check ---
print('=== RAW INPUT RESOLUTIONS ===')
raw_files = {
    'CISI label (target)':     r'READY_data\labels\2024_CISI_010deg_nearest.tif',
    'GDP training (2019)':     r'READY_data\inputs\2019_gdp_aligned_010.tif',
    'Pop training (2020)':     r'READY_data\inputs\2020_pop_aligned_010.tif',
    'Land cover (2020)':       r'READY_data\landuse_onehot\clipped_history_2020_onehot.tif',
    'GDP SSP raw (SSP1 2030)': r'READY_data\GDP SSP\GDP2030_ssp1.tif',
    'Pop SSP raw (SSP1 2030)': r'READY_data\SSP1\SSP1_2030_EU_UK_POP_01.tif',
}
print(f'  {"File":<35} {"Shape":>12} {"Res (°)":>10}  {"0.1° match?"}')
print('  ' + '-' * 75)
for label, path in raw_files.items():
    if not os.path.exists(path):
        print(f'  {label:<35} FILE NOT FOUND')
        continue
    with rasterio.open(path) as src:
        res_x = abs(src.transform[0])
        shape = f'{src.height}x{src.width}'
    ok = '--"' if abs(res_x - 0.1) < 0.001 else f'FAIL  ({res_x:.5f}° --" will be resampled)'
    print(f'  {label:<35} {shape:>12} {res_x:>10.5f}  {ok}')

# --- All 30 normalized output files ---
print('\n=== NORMALIZED OUTPUT FILES (all 30) ===')
print(f'  {"File":<40} {"Shape":>12} {"Res (°)":>10}  {"Status"}')
print('  ' + '-' * 80)

all_ok = True
for ssp in _SSPS:
    for year in _YEARS:
        for kind in ['POP', 'GDP']:
            fname = f'{kind}_{ssp}_{year}_normalized.tif'
            path  = os.path.join(_NORM_DIR, fname)
            if not os.path.exists(path):
                print(f'  {fname:<40} MISSING')
                all_ok = False
                continue
            with rasterio.open(path) as src:
                res_x = abs(src.transform[0])
                shape = f'{src.height}x{src.width}'
            ok = '--"' if abs(res_x - 0.1) < 0.001 else f'wrong res: {res_x:.5f}°'
            print(f'  {fname:<40} {shape:>12} {res_x:>10.5f}  {ok}')
            if abs(res_x - 0.1) >= 0.001:
                all_ok = False

print()
print('All 30 files present and at 0.1°.' if all_ok else 'WARNING: some files are missing or at wrong resolution.')

=== RAW INPUT RESOLUTIONS ===
  File                                       Shape    Res (°)  0.1° match?
  ---------------------------------------------------------------------------
  CISI label (target)                      485x570    0.10000  --"
  GDP training (2019)                      485x570    0.10000  --"
  Pop training (2020)                      485x570    0.10000  --"
  Land cover (2020)                        485x570    0.10000  --"
  GDP SSP raw (SSP1 2030)              18000x43200    0.00833  FAIL  (0.00833° --" will be resampled)
  Pop SSP raw (SSP1 2030)                4554x8480    0.00833  FAIL  (0.00833° --" will be resampled)

=== NORMALIZED OUTPUT FILES (all 30) ===
  File                                            Shape    Res (°)  Status
  --------------------------------------------------------------------------------
  POP_SSP1_2030_normalized.tif                  485x570    0.10000  --"
  GDP_SSP1_2030_normalized.tif                  485x570    0.10000  --"
 

## 5. Process GDP files (15 files)

In [6]:
GDP_FILES = {
    'SSP1': {2030: r'READY_data\GDP SSP\GDP2030_ssp1.tif',
             2050: r'READY_data\GDP SSP\GDP2050_ssp1.tif',
             2100: r'READY_data\GDP SSP\GDP2100_ssp1.tif'},
    'SSP2': {2030: r'READY_data\GDP SSP\GDP2030_ssp2.tif',
             2050: r'READY_data\GDP SSP\GDP2050_ssp2.tif',
             2100: r'READY_data\GDP SSP\GDP2100_ssp2.tif'},
    'SSP3': {2030: r'READY_data\GDP SSP\GDP2030_ssp3.tif',
             2050: r'READY_data\GDP SSP\GDP2050_ssp3.tif',
             2100: r'READY_data\GDP SSP\GDP2100_ssp3.tif'},
    'SSP4': {2030: r'READY_data\GDP SSP\GDP2030_ssp4.tif',
             2050: r'READY_data\GDP SSP\GDP2050_ssp4.tif',
             2100: r'READY_data\GDP SSP\GDP2100_ssp4.tif'},
    'SSP5': {2030: r'READY_data\GDP SSP\GDP2030_ssp5.tif',
             2050: r'READY_data\GDP SSP\GDP2050_ssp5.tif',
             2100: r'READY_data\GDP SSP\GDP2100_ssp5.tif'},
}

gdp_train_mean, gdp_train_std, gdp_train_max = load_training_stats('gdp')
gdp_unit_scale = compute_unit_scale(GDP_FILES, 'gdp', gdp_train_mean)

for ssp, years in GDP_FILES.items():
    for year, in_path in years.items():
        out_path = os.path.join(OUTPUT_DIR, 'GDP_' + ssp + '_' + str(year) + '_normalized.tif')
        normalize_and_save(in_path, out_path, 'gdp', gdp_train_mean, gdp_train_std,
                            unit_scale=gdp_unit_scale)

print('All GDP files processed.')


Training stats for GDP: mean=0.7986  std=0.9003  raw_max=101.49
Computing unit scale for GDP across 15 files...
  SSP log1p mean=3.4486  train log1p mean=0.7986  scale=0.070656
  GDP2030_ssp1.tif  mean=0.6731  range=[-0.887, 4.255]
  GDP2050_ssp1.tif  mean=0.6813  range=[-0.887, 4.255]
  GDP2100_ssp1.tif  mean=0.6904  range=[-0.887, 4.255]
  GDP2030_ssp2.tif  mean=0.674  range=[-0.887, 4.255]
  GDP2050_ssp2.tif  mean=0.6818  range=[-0.887, 4.255]
  GDP2100_ssp2.tif  mean=0.6953  range=[-0.887, 4.255]
  GDP2030_ssp3.tif  mean=0.6733  range=[-0.887, 4.255]
  GDP2050_ssp3.tif  mean=0.6762  range=[-0.887, 4.255]
  GDP2100_ssp3.tif  mean=0.6761  range=[-0.887, 4.255]
  GDP2030_ssp4.tif  mean=0.6733  range=[-0.887, 4.255]
  GDP2050_ssp4.tif  mean=0.6812  range=[-0.887, 4.255]
  GDP2100_ssp4.tif  mean=0.6895  range=[-0.887, 4.255]
  GDP2030_ssp5.tif  mean=0.6764  range=[-0.887, 4.255]
  GDP2050_ssp5.tif  mean=0.6893  range=[-0.887, 4.255]
  GDP2100_ssp5.tif  mean=0.7071  range=[-0.887, 4.255]

In [7]:
NORM_STATS = {
    'gdp': {'train_mean': gdp_train_mean, 'train_std': gdp_train_std,
            'unit_scale': gdp_unit_scale},
    'pop': {'train_mean': pop_train_mean, 'train_std': pop_train_std,
            'train_max': pop_train_max, 'unit_scale': 1.0},
}
stats_path = r'READY_data/inputs_normalized/NORM_STATS_SSP.json'
with open(stats_path, 'w') as f:
    json.dump(NORM_STATS, f, indent=2)
print('Saved', stats_path)


Saved READY_data/inputs_normalized/NORM_STATS_SSP.json


## 6. Verify GDP outputs

In [8]:
_NORM_DIR = r'READY_data\inputs_normalized\ssp'
_SSPS  = ['SSP1', 'SSP2', 'SSP3', 'SSP4', 'SSP5']
_YEARS = [2030, 2050, 2100]

print(f'{"File":<35} {"Shape":>12} {"Valid px":>10} {"Min":>8} {"Max":>8} {"Mean":>8}  {"OK?"}')
print('-' * 100)

all_ok = True
for ssp in _SSPS:
    for year in _YEARS:
        fname = f'GDP_{ssp}_{year}_normalized.tif'
        path  = os.path.join(_NORM_DIR, fname)
        if not os.path.exists(path):
            print(f'{fname:<35} MISSING')
            all_ok = False
            continue
        with rasterio.open(path) as src:
            data  = src.read(1).astype(np.float32)
            res_x = abs(src.transform[0])
            shape = f'{src.height}x{src.width}'
        valid = ~np.isnan(data)
        res_ok    = abs(res_x - 0.1) < 0.001
        has_data  = valid.sum() > 0
        no_extremes = np.all(np.abs(data[valid]) < 20) if has_data else False
        ok = 'OK' if (res_ok and has_data and no_extremes) else 'FAIL'
        if not (res_ok and has_data and no_extremes):
            all_ok = False
        print(f'{fname:<35} {shape:>12} {valid.sum():>10,} {np.nanmin(data):>8.3f} {np.nanmax(data):>8.3f} {np.nanmean(data):>8.4f}  {ok}')

print()
print('All GDP files OK.' if all_ok else 'WARNING: one or more GDP files have issues.')

File                                       Shape   Valid px      Min      Max     Mean  OK?
----------------------------------------------------------------------------------------------------
GDP_SSP1_2030_normalized.tif             485x570    276,450   -0.887    4.255   0.6731  OK
GDP_SSP1_2050_normalized.tif             485x570    276,450   -0.887    4.255   0.6813  OK
GDP_SSP1_2100_normalized.tif             485x570    276,450   -0.887    4.255   0.6904  OK
GDP_SSP2_2030_normalized.tif             485x570    276,450   -0.887    4.255   0.6740  OK
GDP_SSP2_2050_normalized.tif             485x570    276,450   -0.887    4.255   0.6818  OK
GDP_SSP2_2100_normalized.tif             485x570    276,450   -0.887    4.255   0.6953  OK
GDP_SSP3_2030_normalized.tif             485x570    276,450   -0.887    4.255   0.6733  OK
GDP_SSP3_2050_normalized.tif             485x570    276,450   -0.887    4.255   0.6762  OK
GDP_SSP3_2100_normalized.tif             485x570    276,450   -0.887    4.255  

## 7. Process land cover files (15 scenarios x 7 classes = 105 files)

In [9]:
LC_BASE = r'Global 7-land-types LULC projection dataset under SSPs-RCPs'

LC_FILES = {
    'SSP1': {'rcp': 'RCP26', 'years': {2030: 'global_SSP1_RCP26_2030.tif', 2050: 'global_SSP1_RCP26_2050.tif', 2100: 'global_SSP1_RCP26_2100.tif'}},
    'SSP2': {'rcp': 'RCP45', 'years': {2030: 'global_SSP2_RCP45_2030.tif', 2050: 'global_SSP2_RCP45_2050.tif', 2100: 'global_SSP2_RCP45_2100.tif'}},
    'SSP3': {'rcp': 'RCP70', 'years': {2030: 'global_SSP3_RCP70_2030.tif', 2050: 'global_SSP3_RCP70_2050.tif', 2100: 'global_SSP3_RCP70_2100.tif'}},
    'SSP4': {'rcp': 'RCP60', 'years': {2030: 'global_SSP4_RCP60_2030.tif', 2050: 'global_SSP4_RCP60_2050.tif', 2100: 'global_SSP4_RCP60_2100.tif'}},
    'SSP5': {'rcp': 'RCP85', 'years': {2030: 'global_SSP5_RCP85_2030.tif', 2050: 'global_SSP5_RCP85_2050.tif', 2100: 'global_SSP5_RCP85_2100.tif'}},
}


def process_lc(in_path, out_dir, ssp, year):
    """Reproject, clip to Europe, one-hot encode, save 7 class files."""
    print(f'Processing LC: {os.path.basename(in_path)}')

    # Step 1: reproject to EPSG:4326 at 0.10 deg, clipped to Europe
    dst_crs = ref_crs
    dst_transform = ref_transform
    dst_h, dst_w = ref_height, ref_width

    data = np.zeros((dst_h, dst_w), dtype=np.uint8)

    with rasterio.open(in_path) as src:
        rio_reproject(
            source=rasterio.band(src, 1),
            destination=data,
            dst_transform=dst_transform,
            dst_crs=dst_crs,
            resampling=RioResampling.nearest,  # categorical data -- nearest neighbour
        )

    # Step 2: one-hot encode -- save one file per class (1-7)
    profile = ref_profile.copy()
    profile.update(dtype='float32', count=1, nodata=np.nan)

    for c in range(1, 8):
        out_path = os.path.join(out_dir, f'LC_{ssp}_{year}_class_{c}.tif')
        if os.path.exists(out_path):
            print(f'  Skipping class {c} (already exists)')
            continue
        class_mask = (data == c).astype(np.float32)
        # set pixels where data==0 (nodata/ocean) to NaN
        class_mask[data == 0] = np.nan
        with rasterio.open(out_path, 'w', **profile) as dst:
            dst.write(class_mask[np.newaxis, :, :])
        print(f'  Saved class {c} -> {out_path}')


for ssp, info in LC_FILES.items():
    rcp = info['rcp']
    folder = os.path.join(LC_BASE, f'{ssp}_{rcp}')
    for year, fname in info['years'].items():
        in_path = os.path.join(folder, fname)
        if not os.path.exists(in_path):
            print(f'MISSING: {in_path}')
            continue
        # skip if all 7 classes already done
        already = all(os.path.exists(os.path.join(OUTPUT_DIR, f'LC_{ssp}_{year}_class_{c}.tif')) for c in range(1,8))
        if already:
            print(f'Skipping {ssp} {year} LC (all 7 classes exist)')
            continue
        process_lc(in_path, OUTPUT_DIR, ssp, year)

print('All LC files processed.')

Skipping SSP1 2030 LC (all 7 classes exist)
Skipping SSP1 2050 LC (all 7 classes exist)
Skipping SSP1 2100 LC (all 7 classes exist)
Skipping SSP2 2030 LC (all 7 classes exist)
Skipping SSP2 2050 LC (all 7 classes exist)
Skipping SSP2 2100 LC (all 7 classes exist)
Skipping SSP3 2030 LC (all 7 classes exist)
Skipping SSP3 2050 LC (all 7 classes exist)
Skipping SSP3 2100 LC (all 7 classes exist)
Skipping SSP4 2030 LC (all 7 classes exist)
Skipping SSP4 2050 LC (all 7 classes exist)
Skipping SSP4 2100 LC (all 7 classes exist)
Skipping SSP5 2030 LC (all 7 classes exist)
Skipping SSP5 2050 LC (all 7 classes exist)
Skipping SSP5 2100 LC (all 7 classes exist)
All LC files processed.
